# DistilBETO — Nested CV 5×3 con grupos · Optuna TPE20 · curvas y checkpoints


## Revisión antes de la corrida larga

Esta versión incorpora dos correcciones de robustez:

1. al reanudar un checkpoint se restaura también el estado de callbacks, incluyendo el contador de `EarlyStoppingCallback`;
2. los trials huérfanos de Optuna se cierran con la API pública `Study.tell()` en lugar de acceder a `_storage`.

La agrupación anti-leakage se conserva: las equivalencias por texto original, BASE, SOCIAL y tokenización de DistilBETO pertenecen siempre al mismo `GROUP_ID`, y `StratifiedGroupKFold` impide que ese grupo cruce train/validación dentro de un mismo split.

**Nota de comparabilidad:** este experimento usa GroupNestedCV, 20 trials, TPE adaptativo y un espacio de búsqueda ampliado. Por tanto, constituye una segunda iteración experimental y no es exactamente el mismo procedimiento que la corrida previa de 10 trials.


## 1. Instalación y entorno

En la terminal del proyecto (mismo entorno que usa Jupyter):

```powershell
py -m uv add torch transformers accelerate optuna scikit-learn pandas numpy ipython
```

Si ya tienes las dependencias, no necesitas reinstalarlas. Este notebook **no instala paquetes ni inicia GPU** por sí solo al abrirlo.

In [ ]:
import os
import re
# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import hashlib
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)
from IPython.display import display, HTML

import matplotlib
matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer,
    TrainerCallback, EarlyStoppingCallback, set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| optuna", optuna.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 2. Configuración principal


In [ ]:
MODEL_NAME = "dccuchile/distilbert-base-spanish-uncased"
DATA_DIR = Path("../data/transformers")
RESULTS_DIR = Path("../data/resultados_distilbeto_grouped_5x3_tpe20_curvas_ckpts")
TEXT_COLS = {"BASE": "MESSAGE_TRANSFORMER_BASE", "SOCIAL": "MESSAGE_TRANSFORMER_SOCIAL"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"
MAX_LENGTH = 128
SEED = 42

MODO = "prueba"                # Cambiar por "completo" solo después de la prueba.
if MODO == "prueba":
    VARIANTES, N_OUTER, N_INNER, N_TRIALS = ["mx"], 2, 2, 2
elif MODO == "completo":
    VARIANTES, N_OUTER, N_INNER, N_TRIALS = ["mx", "es", "cu"], 5, 3, 20
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

EJECUTAR_NESTED = False           # Seguridad: activa solo después del pre-flight
EJECUTAR_OPTIMIZACION_FINAL = False
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""       # Cuando llegue el momento: "TEST_OFICIAL_CONGELADO"
SEMILLAS_FINALES = [42, 52, 62, 72, 82]
HEARTBEAT_SEGUNDOS = 30
SAVE_CADA_EPOCA = True   # Recuperación: guardar estado al final de cada época.
MANTENER_CHECKPOINTS_COMPLETADOS = False  # Ahorrar disco; nunca borra resultados/curvas.

# Optuna TPE: 5 trials iniciales aleatorios y el resto adaptativos.
# En MODO="prueba" solo se ejecutan 2 trials (ambos exploratorios).
N_STARTUP_TRIALS = 5
SEARCH_SPACE = {
    "preprocessing": ["BASE", "SOCIAL"],
    # 2e-5–5e-5 está respaldado por la literatura revisada.\n    # 6e-5 se incluye como extensión exploratoria porque 5e-5 apareció\n    # repetidamente en el límite superior del experimento anterior.\n    "learning_rate": [2e-5, 3e-5, 4e-5, 5e-5, 6e-5],
    "batch_size": [16, 32],
    "num_train_epochs": [3, 4, 5],
    "weight_decay": [0.0, 0.01, 0.05, 0.1],
    "loss_mode": ["standard", "balanced"],
    "seq_classif_dropout": [0.2, 0.3],
}

# DistilBERT emplea seq_classif_dropout para la cabeza clasificadora.
# En este NUEVO experimento se prueba 0.2 frente a 0.3; BETO es diferente.
RUN_DIR = RESULTS_DIR / "seguimiento" / MODO
TMP_DIR = RESULTS_DIR / "tmp" / MODO
CURVES_DIR = RESULTS_DIR / "curvas" / MODO
FITS_DIR = RESULTS_DIR / "checkpoints" / MODO
RUN_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)
FITS_DIR.mkdir(parents=True, exist_ok=True)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

print(f"[CONFIG][DistilBETO] modo={MODO} | variantes={VARIANTES}")
print(f"[CONFIG][DistilBETO] Nested={N_OUTER}×{N_INNER} | trials={N_TRIALS} | startup={N_STARTUP_TRIALS}")
print(f"[CONFIG][DistilBETO] fine-tunings teóricos={len(VARIANTES) * N_OUTER * (N_INNER * N_TRIALS + 1)}")
print(f"[CONFIG][DistilBETO] nested_activo={EJECUTAR_NESTED} | RUN_DIR={RUN_DIR}")

## 3. Control inicial de los CSV de preprocessing

**Solo se cargan los train** en Nested CV. Si faltan los CSV, ejecuta primero tu notebook final de preprocessing DL corregido. Se usa exactamente el mismo dataset procesado que para BETO.

In [ ]:
def cargar_train(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta el preprocessing DL corregido.")
    df = pd.read_csv(ruta)
    requeridas = [LABEL_COL, "MESSAGE", *TEXT_COLS.values()]
    faltan = [x for x in requeridas if x not in df.columns]
    if faltan:
        raise ValueError(f"Columnas ausentes en {ruta}: {faltan}")
    if df[list(TEXT_COLS.values())].isna().any().any():
        raise ValueError(f"{variante}: BASE/SOCIAL tienen textos vacíos por NaN; revisar preprocessing.")
    if df.duplicated(subset="MESSAGE").any():
        raise ValueError(f"{variante}: hay MESSAGE duplicados en el train depurado.")
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    if set(df[LABEL_COL].unique()) != {0, 1}:
        raise ValueError(f"{variante}: se esperaban exactamente las clases 0 y 1.")
    for col in TEXT_COLS.values():
        df[col] = df[col].astype(str)
    return df.reset_index(drop=True)

for variante in VARIANTES:
    df = cargar_train(variante)
    print(variante.upper(), len(df), "muestras", df[LABEL_COL].value_counts().to_dict())

## 4. Tokenizador oficial y datos

Se usa el **tokenizer propio de DistilBETO uncased**, que se encarga de pasar el texto a minúsculas. No se elimina puntuación, tildes ni stopwords en esta fase. Se tokenizan previamente tanto `BASE` como `SOCIAL` (sin usar etiquetas para crear el vocabulario) y cada partición utiliza índices del mismo dataframe. Padding dinámico por batch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Modelo uncased:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

## 4A. Auditoría y construcción de `GROUP_ID`

Se construyen grupos de equivalencia de forma independiente para DistilBETO usando el texto original, BASE, SOCIAL y las secuencias tokenizadas con truncamiento a 128. Los registros equivalentes permanecen juntos mediante `StratifiedGroupKFold`; no se elimina ningún registro por esta auditoría.


In [ ]:

from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def hash_tokens(ids):
    texto = ",".join(map(str, ids))
    return hashlib.sha1(texto.encode("utf-8")).hexdigest()


def construir_grupos_distilbeto(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    La agrupación es propia de DistilBETO: texto original, BASE, SOCIAL y
    secuencias tokenizadas (uncased + truncamiento a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    trabajo["KEY_BASE"] = trabajo[TEXT_COLS["BASE"]].map(colapsar_espacios)
    trabajo["KEY_SOCIAL"] = trabajo[TEXT_COLS["SOCIAL"]].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True, max_length=max_length, padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [hash_tokens(x) for x in ids]

    key_cols = ["KEY_ORIGINAL", "KEY_BASE", "KEY_SOCIAL", "KEY_TOK_BASE", "KEY_TOK_SOCIAL"]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)
    for miembros in componentes.values():
        if ID_COL in trabajo.columns:
            ids = sorted(trabajo.loc[miembros, ID_COL].astype(str).tolist())
        else:
            ids = [str(x) for x in sorted(miembros)]
        digest = hashlib.sha1("|".join(ids).encode("utf-8")).hexdigest()[:12]
        gid = f"{variante}_{digest}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in [
        ("ORIGINAL", "KEY_ORIGINAL"),
        ("BASE", "KEY_BASE"),
        ("SOCIAL", "KEY_SOCIAL"),
        ("TOK_BASE", "KEY_TOK_BASE"),
        ("TOK_SOCIAL", "KEY_TOK_SOCIAL"),
    ]:
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append({
            "variante": variante,
            "representacion": nombre,
            "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
            "grupos_repetidos": int(len(repetidos)),
        })

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se auditan las tres variantes antes de una corrida larga, incluso en MODO="prueba".
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_distilbeto(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RESULTS_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(
        RESULTS_DIR / "auditoria_grupos_labels_mixtos.csv", index=False
    )

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][DistilBETO][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )


## 4B. Pre-flight metodológico


In [ ]:
def comprobar_preflight_distilbeto():
    print("=" * 78)
    print("PRE-FLIGHT — DISTILBETO GROUPED NESTED CV")
    print("=" * 78)

    if MODO == "completo" and (N_OUTER, N_INNER, N_TRIALS, N_STARTUP_TRIALS) != (5, 3, 20, 5):
        raise ValueError("Configuración completa inesperada; se esperaba Nested 5×3, 20 trials y startup=5.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA no disponible. Verifica entorno/PyTorch antes de entrenar DistilBETO.")

    for variante in ["mx", "es", "cu"]:
        df = TRAIN_AGRUPADO[variante]
        if df[ID_COL].duplicated().any():
            raise ValueError(f"{variante.upper()}: IDs duplicados en train procesado.")
        if df[GROUP_COL].isna().any():
            raise ValueError(f"{variante.upper()}: GROUP_ID nulo.")
        if not set(df[LABEL_COL].unique()).issubset({0, 1}):
            raise ValueError(f"{variante.upper()}: etiquetas fuera de 0/1.")
        n_mixtos = int(df.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
        if n_mixtos:
            print(
                f"[AUDIT][DistilBETO][{variante.upper()}] ADVERTENCIA: "
                f"{n_mixtos} GROUP_ID con etiquetas mixtas; no es leakage porque el grupo no se divide, "
                "pero debe documentarse como ruido/ambigüedad de anotación."
            )

    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Nested: {N_OUTER}×{N_INNER} | Optuna={N_TRIALS} | startup={N_STARTUP_TRIALS}")
    print("PRE-FLIGHT DE DATOS: OK")


comprobar_preflight_distilbeto()


## 5. Métricas, pesos de clase y modelos independientes

En cada Inner Fold la pérdida balanceada calcula pesos **solo** con el train de ese fold. Cada llamada `crear_modelo()` carga DistilBETO desde cero con nueva cabeza de clasificación binaria. Los mensajes de pesos `MISSING`/`UNEXPECTED` quedan en un log, no en Jupyter.

In [ ]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    return {
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


def crear_modelo(seed, seq_classif_dropout=0.2):
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, \
         redirect_stdout(log), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "distilbert":
            raise ValueError(f"Se esperaba arquitectura DistilBERT, recibida: {config.model_type}")
        # DistilBERT: dropout de la cabeza de clasificación, no classifier_dropout de BERT.
        config.seq_classif_dropout = float(seq_classif_dropout)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model


class ClasificadorTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        self.class_weights = class_weights
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        if self.class_weights is None:
            loss = outputs.loss
        else:
            weights = self.class_weights.to(outputs.logits.device)
            loss = F.cross_entropy(outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss

## 6. Argumentos de entrenamiento

El Trainer evalúa por época, guarda el mejor checkpoint por F1-Macro, utiliza AdamW + 10% warmup y detiene entrenamiento si F1 no mejora durante una evaluación. En el **refit externo y final no hay early stopping**, porque no se puede usar el Outer Test ni el test oficial como validación.

In [ ]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "warmup_ratio": 0.10,
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "save_safetensors": True,
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        params.update({"metric_for_best_model": "f1_macro", "greater_is_better": True})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    return TrainingArguments(**{k: v for k, v in params.items() if k in sig})


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)


## 7. Monitor único y archivos de recuperación

El panel se actualiza sin crear cientos de outputs. Se registran señales en `monitor.log` aproximadamente cada 30 segundos durante los pasos de entrenamiento. Los trials completos se guardan en una base SQLite por Outer Fold y los resultados de cada Outer Fold se escriben inmediatamente en CSV.

Si un entrenamiento se interrumpe, puedes ejecutar otra vez el notebook con **los mismos CSV y la misma configuración**. Se comprueba una firma SHA256 de los CSV para evitar mezclar experimentos distintos.

In [ ]:
class Monitor:
    def __init__(self, ruta, total, ya_completados=0):
        self.ruta = Path(ruta)
        self.ruta.mkdir(parents=True, exist_ok=True)
        self.log = self.ruta / "monitor.log"
        self.total = int(total)
        self.completados = int(ya_completados)
        self.completados_inicio = int(ya_completados)
        self.inicio = time.monotonic()
        self.ultimo_pulso = 0
        self.etapa = "Inicializando"
        self.handle = display(HTML("<b>Preparando DistilBETO...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def registrar(self, msg):
        with self.log.open("a", encoding="utf-8") as f:
            f.write(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n")

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = etapa
        ahora = time.monotonic()
        if not forzar and ahora - self.ultimo_pulso < HEARTBEAT_SEGUNDOS:
            return
        self.ultimo_pulso = ahora
        minutos = (ahora - self.inicio) / 60
        hechos_sesion = self.completados - self.completados_inicio
        faltan = max(0, self.total - self.completados)
        eta = (f"~{minutos / hechos_sesion * faltan:.0f} min" if hechos_sesion >= 2
               else "calculando...")
        marca = datetime.now().strftime("%H:%M:%S")
        panel = (
            '<div style="font-family:system-ui;padding:12px;border:1px solid #777;'
            'border-radius:8px;max-width:800px">'
            '<b>DistilBETO — seguimiento en vivo</b><br>'
            f'Etapa: <b>{html.escape(self.etapa)}</b><br>'
            f'Fine-tunings completados: {self.completados}/{self.total} '
            f'({100*self.completados/max(1,self.total):.1f}%)<br>'
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f'Tiempo: {minutos:.1f} min | ETA orientativo: {eta}<br>'
            f'Última señal: {marca}<br>Log: {html.escape(str(self.log))}</div>'
        )
        self.handle.update(HTML(panel))

    def iniciar(self, msg):
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def pulso(self):
        if time.monotonic() - self.ultimo_pulso >= HEARTBEAT_SEGUNDOS:
            self.registrar("ACTIVO: " + self.etapa)
            self.mostrar(forzar=True)

    def fit_terminado(self, msg):
        self.completados += 1
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)


class PulsoTrainer(TrainerCallback):
    def __init__(self, monitor):
        self.monitor = monitor

    def on_step_end(self, args, state, control, **kwargs):
        self.monitor.pulso()
        return control


def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


def firma_actual():
    return {
        "checkpoint": MODEL_NAME, "modo": MODO, "variantes": VARIANTES,
        "outer": N_OUTER, "inner": N_INNER, "trials": N_TRIALS,
        "startup_trials": N_STARTUP_TRIALS,
        "semilla": SEED, "max_length": MAX_LENGTH, "search_space": SEARCH_SPACE,
        "group_strategy": "original_base_social_tokenized_connected_components_v1",
        "fixed_training": {
            "optim": "adamw_torch", "warmup_ratio": 0.10,
            "early_stopping_patience": 1,
            "refit_epoch_strategy": "median_inner_best_epoch_rounded",
            "curvas": "train_eval_loss_epoch_csv_best_outer_png_v1",
            "checkpoint": "epoch_inner_and_refit_save_total_limit_2_v1",
        },
        "data": {
            v: hashlib.sha256((DATA_DIR / f"train_transformers_{v}.csv").read_bytes()).hexdigest()
            for v in VARIANTES
        },
    }


def verificar_firma():
    ruta = RUN_DIR / "config_ejecucion.json"
    actual = firma_actual()
    if ruta.exists():
        anterior = json.loads(ruta.read_text(encoding="utf-8"))
        if anterior != actual:
            raise ValueError(
                "Esta carpeta tiene una corrida con datos/configuración diferentes. "
                "No mezcles resultados; cambia MODO/carpeta o archiva el experimento."
            )
    else:
        tmp = ruta.with_suffix(".tmp")
        tmp.write_text(json.dumps(actual, indent=2, ensure_ascii=False), encoding="utf-8")
        os.replace(tmp, ruta)


def ruta_estudio(variante, nombre):
    return RUN_DIR / f"distilbeto_{variante}_{nombre}.db"


def cargar_study(variante, nombre):
    ruta = ruta_estudio(variante, nombre)
    if not ruta.exists():
        return None
    return optuna.load_study(
        study_name=f"DistilBETO_{variante}_{nombre}",
        storage=f"sqlite:///{ruta.resolve().as_posix()}",
    )


def contar_completados():
    n = 0
    for variante in VARIANTES:
        ruta_csv = RUN_DIR / f"distilbeto_detalle_{variante}.csv"
        completados = set()
        if ruta_csv.exists():
            completados = set(pd.read_csv(ruta_csv)["outer_fold"].astype(int))
        for outer in range(1, N_OUTER + 1):
            if outer in completados:
                n += N_TRIALS * N_INNER + 1
            else:
                st = cargar_study(variante, f"outer{outer}")
                if st is not None:
                    n += min(N_TRIALS, sum(
                        t.state == optuna.trial.TrialState.COMPLETE for t in st.trials
                    )) * N_INNER
    return n

## 7A. Checkpoints por época y curvas de pérdida

Las carpetas de cada entrenamiento se identifican por **variante, Outer Fold, configuración exacta e Inner Fold**, no por número de trial: si Optuna debe reencolar un trial interrumpido, se pueden recuperar sus Inner Folds sin perderlos. El estado se guarda al finalizar cada época, nunca en el test oficial.

Los CSV contienen `epoch`, `train_loss`, `val_loss` y `val_f1_macro`. Se generan gráficos por Outer Fold del **mejor trial según Inner CV**, con las tres particiones internas; al acabar cada variante se puede consultar la carpeta `curvas/<modo>/`. Las épocas que no alcanzan todos los folds por early stopping se indican con `n_folds` en el CSV agregado.


In [ ]:
# ==============================================================
# CHECKPOINTS, CURVAS Y REANUDACIÓN A NIVEL DE INNER FOLD
# ==============================================================
def firma_config(cfg):
    blob = json.dumps(cfg, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def carpeta_inner(variante, nombre, cfg, fold):
    return FITS_DIR / variante / nombre / firma_config(cfg) / f"inner{fold}"


def carpeta_refit(variante, outer_fold):
    return FITS_DIR / variante / f"outer{outer_fold}" / "refit"


def json_atomico(obj, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".tmp")
    temporal.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporal, ruta)


def checkpoints_validos(ruta):
    """Evita reanudar desde un checkpoint escrito solo parcialmente."""
    base = Path(ruta)
    candidatos = []
    for ckpt in base.glob("checkpoint-*"):
        try:
            step = int(ckpt.name.rsplit("-", 1)[1])
        except (ValueError, IndexError):
            continue
        modelo = any((ckpt / nombre).exists() for nombre in
                     ("model.safetensors", "pytorch_model.bin"))
        # Si faltan optimizador, scheduler o estado, no es seguro reanudar.
        requerido = all((ckpt / nombre).exists() for nombre in
                       ("trainer_state.json", "optimizer.pt", "scheduler.pt"))
        if modelo and requerido:
            candidatos.append((step, ckpt))
    return [p for _, p in sorted(candidatos, reverse=True)]


def ultimo_checkpoint_valido(ruta):
    candidatos = checkpoints_validos(ruta)
    return str(candidatos[0]) if candidatos else None


def extraer_curva(log_history):
    """Empareja train/validation por número de época, sin tocar Outer Test."""
    filas = {}
    for entry in log_history:
        if "epoch" not in entry:
            continue
        epoch = int(round(float(entry["epoch"])))
        if epoch < 1:
            continue
        fila = filas.setdefault(epoch, {"epoch": epoch,
                                      "train_loss": np.nan,
                                      "val_loss": np.nan,
                                      "val_f1_macro": np.nan})
        if "loss" in entry:
            fila["train_loss"] = float(entry["loss"])
        if "eval_loss" in entry:
            fila["val_loss"] = float(entry["eval_loss"])
        if "eval_f1_macro" in entry:
            fila["val_f1_macro"] = float(entry["eval_f1_macro"])
    return pd.DataFrame(list(filas.values()), columns=[
        "epoch", "train_loss", "val_loss", "val_f1_macro"
    ]).sort_values("epoch").reset_index(drop=True)


def limpiar_checkpoints_completados(carpeta):
    if MANTENER_CHECKPOINTS_COMPLETADOS:
        return
    # Solo los pesos/estados temporales, no resultado.json ni curva.csv.
    for p in Path(carpeta).glob("checkpoint-*"):
        if p.is_dir():
            shutil.rmtree(p)


def graficar_mejor_trial(variante, outer_fold, cfg, n_inner, mostrar=False):
    """Llamar SOLO después de elegir el mejor trial (no usa el Outer Test)."""
    nombre = f"outer{outer_fold}"
    curves = []
    for fold in range(1, n_inner + 1):
        ruta = carpeta_inner(variante, nombre, cfg, fold) / "curva.csv"
        if not ruta.exists():
            raise FileNotFoundError(f"Falta curva del mejor trial: {ruta}")
        d = pd.read_csv(ruta)
        d["inner_fold"] = fold
        curves.append(d)
    todos = pd.concat(curves, ignore_index=True)
    resumen = (todos.groupby("epoch", as_index=False)
               .agg(train_loss=("train_loss", "mean"),
                    val_loss=("val_loss", "mean"),
                    train_std=("train_loss", "std"),
                    val_std=("val_loss", "std"),
                    val_f1_macro=("val_f1_macro", "mean"),
                    n_folds=("inner_fold", "nunique")))
    salida = CURVES_DIR / variante
    salida.mkdir(parents=True, exist_ok=True)
    csv_atomico(todos, salida / f"outer{outer_fold}_inner_folds.csv")
    csv_atomico(resumen, salida / f"outer{outer_fold}_media_por_epoca.csv")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(resumen["epoch"], resumen["train_loss"], marker="o", label="Train loss")
    ax.plot(resumen["epoch"], resumen["val_loss"], marker="o", label="Validation loss")
    ax.set(xlabel="Época", ylabel="Loss",
           title=f"DistilBETO {variante.upper()} — Outer {outer_fold} | mejor trial interno")
    ax.set_xticks(resumen["epoch"].astype(int).tolist())
    ax.grid(alpha=0.25)
    ax.legend()
    # Las medias pueden incluir menos folds en las últimas épocas.
    fig.text(0.13, 0.01, "n_folds por época disponible en CSV; validación INNER (no Outer Test).",
             fontsize=8)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    loss_png = salida / f"outer{outer_fold}_loss.png"
    fig.savefig(loss_png, dpi=160, bbox_inches="tight")
    if mostrar:
        display(fig)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(resumen["epoch"], resumen["val_f1_macro"], marker="o")
    ax.set(
        xlabel="Época",
        ylabel="F1-Macro validación",
        title=f"DistilBETO {variante.upper()} — Outer {outer_fold} | mejor trial interno",
    )
    ax.set_xticks(resumen["epoch"].astype(int).tolist())
    ax.grid(alpha=0.25)
    fig.tight_layout()
    f1_png = salida / f"outer{outer_fold}_f1.png"
    fig.savefig(f1_png, dpi=160, bbox_inches="tight")
    if mostrar:
        display(fig)
    plt.close(fig)

    return {
        "csv_inner": str(salida / f"outer{outer_fold}_inner_folds.csv"),
        "csv_media": str(salida / f"outer{outer_fold}_media_por_epoca.csv"),
        "loss_png": str(loss_png),
        "f1_png": str(f1_png),
    }


## 8. Búsqueda interna: TPE adaptativo con Optuna

Cada estudio (uno por Outer Fold) prueba 20 configuraciones: 5 propuestas iniciales aleatorias y hasta 15 adaptativas. Cada configuración se evalúa con **tres Inner Folds**, y la medida que se maximiza es su F1-Macro promedio. Como el refit usa la mediana de las mejores épocas internas, el Outer Test nunca determina ni las épocas ni los hiperparámetros.

Se mantiene la misma semilla de inicialización por Inner Fold entre trials para reducir ruido en la comparación (cada modelo se vuelve a cargar desde el checkpoint original). El nuevo espacio incluye `seq_classif_dropout=0.2/0.3`, `learning_rate` hasta `6e-5` y `weight_decay` hasta `0.1` para estudiar regularización; las mejoras no están garantizadas.

**Recuperación:** SQLite guarda trials completos, los CSV muestran el historial y un archivo del sampler conserva su estado para continuaciones. Un trial que se corta a mitad de entrenamiento se repetirá completo al recuperar.


In [ ]:
def sugerir_config(trial):
    return {
        "preprocessing": trial.suggest_categorical("preprocessing", SEARCH_SPACE["preprocessing"]),
        "learning_rate": trial.suggest_categorical("learning_rate", SEARCH_SPACE["learning_rate"]),
        "batch_size": trial.suggest_categorical("batch_size", SEARCH_SPACE["batch_size"]),
        "num_train_epochs": trial.suggest_categorical("num_train_epochs", SEARCH_SPACE["num_train_epochs"]),
        "weight_decay": trial.suggest_categorical("weight_decay", SEARCH_SPACE["weight_decay"]),
        "loss_mode": trial.suggest_categorical("loss_mode", SEARCH_SPACE["loss_mode"]),
        "seq_classif_dropout": trial.suggest_categorical(
            "seq_classif_dropout", SEARCH_SPACE["seq_classif_dropout"]),
    }


def buscar_en_train(cache, y, groups, indices, variante, nombre, k_inner, n_trials, seed_cv, monitor=None):
    indices = np.asarray(indices, dtype=int)
    y_inner = y[indices]
    groups_inner = np.asarray(groups)[indices]
    skf = StratifiedGroupKFold(n_splits=k_inner, shuffle=True, random_state=seed_cv)
    splits = list(skf.split(np.zeros(len(indices)), y_inner, groups=groups_inner))

    for numero, (idx_tr_local, idx_val_local) in enumerate(splits, start=1):
        g_tr = set(groups_inner[idx_tr_local])
        g_val = set(groups_inner[idx_val_local])
        if g_tr & g_val:
            raise AssertionError(
                f"{variante} {nombre} Inner {numero}: GROUP_ID compartido entre train y validación."
            )

    def objetivo(trial):
        cfg = sugerir_config(trial)
        ds = cache[cfg["preprocessing"]]
        scores, epocas = [], []
        for numero, (idx_tr_local, idx_val_local) in enumerate(splits, start=1):
            tr, val = indices[idx_tr_local], indices[idx_val_local]
            carpeta = carpeta_inner(variante, nombre, cfg, numero)
            carpeta.mkdir(parents=True, exist_ok=True)
            resultado_file = carpeta / "resultado.json"
            if monitor:
                monitor.iniciar(f"{variante.upper()} {nombre} | trial ID {trial.number} | inner {numero}/{k_inner}")
            # El archivo solo existe si este fold terminó y se guardaron sus curvas.
            if resultado_file.exists():
                previo = json.loads(resultado_file.read_text(encoding="utf-8"))
                if previo.get("config_firma") != firma_config(cfg):
                    raise ValueError("Conflicto entre el checkpoint y configuración de Optuna")
                scores.append(float(previo["f1_inner"]))
                epocas.append(int(previo["best_epoch"]))
                if monitor:
                    monitor.fit_terminado(f"REUTILIZADO {variante.upper()} {nombre} "
                                         f"trial {trial.number} inner {numero} F1={scores[-1]:.4f}")
                continue

            # Los pesos se crean nuevos; la misma semilla por fold evita
            # diferencias de inicialización entre configuraciones.
            seed_fold = SEED + seed_cv * 100 + numero
            model = crear_modelo(seed_fold, cfg["seq_classif_dropout"])
            class_weights = pesos_balanceados(y[tr]) if cfg["loss_mode"] == "balanced" else None
            trainer = None
            try:
                # Checkpoints PERSISTENTES; no TemporaryDirectory, que los borraría en un crash.
                args = argumentos_entrenamiento(carpeta, cfg, seed_fold, con_validacion=True)
                callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]
                if monitor:
                    callbacks.append(PulsoTrainer(monitor))
                trainer = entrenador_silencioso(ClasificadorTrainer(
                    model=model, args=args,
                    train_dataset=Subset(ds, tr.tolist()),
                    eval_dataset=Subset(ds, val.tolist()),
                    data_collator=data_collator, compute_metrics=metricas,
                    callbacks=callbacks, class_weights=class_weights,
                ))
                checkpoint = ultimo_checkpoint_valido(carpeta)
                if checkpoint and monitor:
                    monitor.registrar(f"REANUDANDO: {checkpoint}")
                trainer.train(resume_from_checkpoint=checkpoint)
                valor = trainer.state.best_metric
                if valor is None:
                    raise RuntimeError("No hay mejor F1 de validación en este Inner Fold.")
                f1 = float(valor)
                epoch = mejor_epoca(trainer, cfg["num_train_epochs"])
                curva = extraer_curva(trainer.state.log_history)
                if curva.empty or curva["val_loss"].isna().all() or curva["train_loss"].isna().all():
                    raise RuntimeError("Faltan las curvas train/validation loss.")
                # CSV primero y marcador de finalización SOLO cuando está completo.
                csv_atomico(curva, carpeta / "curva.csv")
                json_atomico({
                    "config_firma": firma_config(cfg),
                    "f1_inner": f1, "best_epoch": epoch,
                    "n_epocas_registradas": len(curva),
                }, resultado_file)
                limpiar_checkpoints_completados(carpeta)
                scores.append(f1)
                epocas.append(epoch)
            finally:
                if trainer is not None:
                    del trainer
                del model
                liberar_memoria()
            if monitor:
                monitor.fit_terminado(f"{variante.upper()} {nombre} trial {trial.number} "
                                     f"inner {numero}: F1={f1:.4f}")

        trial.set_user_attr("f1_inner_std", float(np.std(scores, ddof=1)))
        trial.set_user_attr("best_epochs_inner", [int(e) for e in epocas])
        trial.set_user_attr("refit_epochs", int(np.clip(round(np.median(epocas)), 1, cfg["num_train_epochs"])))
        return float(np.mean(scores))

    sampler_file = RUN_DIR / f"sampler_{variante}_{nombre}.pkl"
    if sampler_file.exists():
        # Solo pickle propio del mismo experimento, jamás un archivo de terceros.
        with sampler_file.open("rb") as f:
            sampler = pickle.load(f)
    else:
        sampler = optuna.samplers.TPESampler(seed=seed_cv, n_startup_trials=N_STARTUP_TRIALS)
    study = optuna.create_study(
        direction="maximize", sampler=sampler,
        study_name=f"DistilBETO_{variante}_{nombre}",
        storage=f"sqlite:///{ruta_estudio(variante, nombre).resolve().as_posix()}",
        load_if_exists=True,
    )

    # Si se apagó el equipo en medio de un trial, SQLite deja RUNNING.
    # Se encola la misma configuración y se recupera su Inner Fold parcial.
    # Los trials huérfanos se cierran con Study.tell(..., FAIL), API pública de Optuna.
    running = [t for t in study.trials if t.state == optuna.trial.TrialState.RUNNING]
    waiting = [t for t in study.trials if t.state == optuna.trial.TrialState.WAITING]
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    for interrumpido in running:
        ya_reencolado = any(
            (t.state == optuna.trial.TrialState.WAITING and
             t.system_attrs.get("fixed_params", {}) == interrumpido.params)
            or (t.state == optuna.trial.TrialState.COMPLETE and t.params == interrumpido.params)
            for t in waiting + done
        )
        if not ya_reencolado and interrumpido.params:
            study.enqueue_trial(interrumpido.params)
        try:
            # API pública de Optuna: cerrar limpiamente el trial huérfano.
            study.tell(
                interrumpido.number,
                state=optuna.trial.TrialState.FAIL,
                skip_if_finished=True,
            )
        except (ValueError, RuntimeError) as exc:
            raise RuntimeError(
                f"No se pudo marcar como FAIL el trial interrumpido "
                f"{interrumpido.number}"
            ) from exc
        if monitor:
            monitor.registrar(f"Trial interrumpido {interrumpido.number} reencolado; retomará folds/épocas")

    # Optuna marca FAIL cuando el usuario detiene Jupyter con KeyboardInterrupt.
    # Si dejó un checkpoint de época o un Inner Fold completo, reencolar la MISMA
    # configuración antes de iniciar trials nuevos (no confundir con error sin progreso).
    fallidos_parciales = []
    for anterior in study.trials:
        if anterior.state != optuna.trial.TrialState.FAIL or not anterior.params:
            continue
        bases = [carpeta_inner(variante, nombre, anterior.params, numero)
                 for numero in range(1, k_inner + 1)]
        tiene_avance = any((base / "resultado.json").exists() or
                           ultimo_checkpoint_valido(base) is not None for base in bases)
        if tiene_avance:
            fallidos_parciales.append(anterior)
    for anterior in fallidos_parciales:
        actuales = study.trials  # WAITING params están en system_attrs.fixed_params.
        ya_pendiente_o_completo = any(
            (t.state == optuna.trial.TrialState.COMPLETE and t.params == anterior.params)
            or (t.state == optuna.trial.TrialState.WAITING and
                t.system_attrs.get("fixed_params", {}) == anterior.params)
            for t in actuales
        )
        if not ya_pendiente_o_completo:
            study.enqueue_trial(anterior.params)
            if monitor:
                monitor.registrar(f"Reencolado trial fallido {anterior.number} "
                                 "con checkpoints parciales; mismos hiperparámetros")

    completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    faltantes = max(0, n_trials - completos)
    if monitor and completos:
        monitor.registrar(f"Recuperados {completos} trials de {variante} {nombre}; faltan {faltantes}.")

    def trial_finalizado(st, trial):
        tmp_sampler = sampler_file.with_suffix(".pkl.tmp")
        with tmp_sampler.open("wb") as f:
            pickle.dump(st.sampler, f)
        os.replace(tmp_sampler, sampler_file)
        csv_atomico(st.trials_dataframe(), RUN_DIR / f"trials_{variante}_{nombre}.csv")
        n_completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in st.trials)
        if trial.state == optuna.trial.TrialState.COMPLETE:
            msg = (f"{variante.upper()} {nombre} | trial {n_completos}/{n_trials} "
                   f"F1_inner={trial.value:.4f} | mejor={st.best_value:.4f}")
        else:
            msg = f"{variante.upper()} {nombre} | trial ID {trial.number} {trial.state.name}"
        if monitor:
            monitor.registrar(msg)
            monitor.mostrar(msg, forzar=True)

    if faltantes:
        study.optimize(objetivo, n_trials=faltantes, callbacks=[trial_finalizado],
                       show_progress_bar=False, gc_after_trial=True)
    completos_final = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if completos_final < n_trials:
        raise RuntimeError("No se completaron todos los trials; consulta monitor.log.")
    return study


## 9. Refit externo sin utilizar Outer Test para early stopping

Se crea un DistilBETO nuevo y se entrena sobre todo `Outer Train`, durante la mediana de las mejores épocas internas. El `Outer Test` se usa solo para calcular F1 y las demás métricas. Se registran F1 train, F1 inner, F1 outer y gap.

In [ ]:
def refit_externo(cache, y, tr, out, cfg, epocas, seed_modelo,
                  monitor=None, variante=None, outer_fold=None):
    if variante is None or outer_fold is None:
        raise ValueError("Refit externo necesita variante y outer_fold para guardar checkpoints")
    carpeta = carpeta_refit(variante, outer_fold)
    carpeta.mkdir(parents=True, exist_ok=True)
    resultado_file = carpeta / "resultado.json"
    cfg_hash = firma_config(cfg)
    if resultado_file.exists():
        previo = json.loads(resultado_file.read_text(encoding="utf-8"))
        if previo.get("config_firma") != cfg_hash:
            raise ValueError("Refit almacenado no coincide con la configuración seleccionada")
        if monitor:
            monitor.fit_terminado(f"REUTILIZADO refit {variante.upper()} outer {outer_fold}")
        return {
            k: float(previo[k])
            for k in ("f1_train", "f1_outer_clase_0", "f1_outer_clase_1", "f1_outer", "gap")
        }

    ds = cache[cfg["preprocessing"]]
    if monitor:
        monitor.iniciar(f"{variante.upper()} refit Outer {outer_fold} | {epocas} épocas")
    model = crear_modelo(seed_modelo, cfg["seq_classif_dropout"])
    cw = pesos_balanceados(y[tr]) if cfg["loss_mode"] == "balanced" else None
    trainer = None
    try:
        args = argumentos_entrenamiento(carpeta, cfg, seed_modelo,
                                       con_validacion=False, epochs=epocas)
        callbacks = [PulsoTrainer(monitor)] if monitor else []
        trainer = entrenador_silencioso(ClasificadorTrainer(
            model=model, args=args, train_dataset=Subset(ds, tr.tolist()),
            data_collator=data_collator, class_weights=cw, callbacks=callbacks,
        ))
        checkpoint = ultimo_checkpoint_valido(carpeta)
        if checkpoint and monitor:
            monitor.registrar(f"REANUDANDO refit: {checkpoint}")
        trainer.train(resume_from_checkpoint=checkpoint)
        # Outer Test se usa UNA VEZ al terminar el refit; nunca para early stopping.
        p_train = predecir(trainer, Subset(ds, tr.tolist()))
        p_outer = predecir(trainer, Subset(ds, out.tolist()))
        f_train = float(f1_score(y[tr], p_train, average="macro"))
        f_outer_clases = f1_score(
            y[out], p_outer, labels=[0, 1], average=None, zero_division=0
        )
        f_outer = float(f1_score(y[out], p_outer, average="macro"))
        curva = extraer_curva(trainer.state.log_history)
        if not curva.empty:
            csv_atomico(curva, carpeta / "train_loss_refit.csv")
        out = {
            "f1_train": f_train,
            "f1_outer_clase_0": float(f_outer_clases[0]),
            "f1_outer_clase_1": float(f_outer_clases[1]),
            "f1_outer": f_outer,
            "gap": f_train - f_outer,
        }
        json_atomico({"config_firma": cfg_hash, **out}, resultado_file)
        limpiar_checkpoints_completados(carpeta)
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
    if monitor:
        monitor.fit_terminado(f"Refit {variante.upper()} Outer {outer_fold} "
                             f"terminado | train={f_train:.4f} outer={f_outer:.4f}")
    return out


## 10. Nested CV por variante (reanudar sin perder Outer Folds)


In [ ]:
def ejecutar_variante(variante, monitor):
    df = TRAIN_AGRUPADO[variante].copy()
    cache, y = cache_dataset(df)
    groups = df[GROUP_COL].astype(str).to_numpy()
    outer_cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
    splits = list(outer_cv.split(np.zeros(len(y)), y, groups=groups))
    for numero, (tr_chk, out_chk) in enumerate(splits, start=1):
        if set(groups[tr_chk]) & set(groups[out_chk]):
            raise AssertionError(f"{variante} Outer {numero}: GROUP_ID compartido train/test.")
    ruta_parcial = RUN_DIR / f"distilbeto_detalle_{variante}.csv"
    if ruta_parcial.exists():
        resultados = pd.read_csv(ruta_parcial).to_dict("records")
        folds_hechos = {int(x["outer_fold"]) for x in resultados}
    else:
        resultados, folds_hechos = [], set()

    for outer_fold, (tr, out) in enumerate(splits, start=1):
        if outer_fold in folds_hechos:
            msg = f"[{variante.upper()}][DistilBETO][Outer {outer_fold}/{N_OUTER}] resultado recuperado; se omite."
            print(msg)
            monitor.registrar(msg)
            continue
        monitor.iniciar(f"{variante.upper()} | Outer {outer_fold}/{N_OUTER}")
        study = buscar_en_train(
            cache, y, groups, tr, variante, f"outer{outer_fold}", N_INNER, N_TRIALS,
            seed_cv=SEED + outer_fold, monitor=monitor,
        )
        cfg = study.best_params.copy()
        refit_epochs = int(study.best_trial.user_attrs["refit_epochs"])
        ext = refit_externo(
            cache, y, tr, out, cfg, refit_epochs,
            seed_modelo=SEED + 1000 + outer_fold, monitor=monitor,
            variante=variante, outer_fold=outer_fold,
        )
        # Graficar solamente las curvas del MEJOR TRIAL INTERNO del Outer Fold.
        graficas = graficar_mejor_trial(variante, outer_fold, cfg, N_INNER, mostrar=False)
        fila = {
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": cfg["preprocessing"],
            "f1_inner": float(study.best_value),
            "std_inner": float(study.best_trial.user_attrs["f1_inner_std"]),
            "refit_epochs": refit_epochs,
            "f1_train": ext["f1_train"],
            "f1_outer_clase_0": ext["f1_outer_clase_0"],
            "f1_outer_clase_1": ext["f1_outer_clase_1"],
            "f1_outer": ext["f1_outer"],
            "gap": ext["gap"],
            "grafica_loss": graficas["loss_png"],
            "grafica_f1": graficas["f1_png"],
            "best_params": json.dumps(cfg, ensure_ascii=False),
        }
        resultados.append(fila)
        csv_atomico(pd.DataFrame(resultados).sort_values("outer_fold"), ruta_parcial)
        msg = (
            f"[{variante.upper()}][DistilBETO][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={fila['best_preprocessing']} | "
            f"F1_inner={fila['f1_inner']:.4f}±{fila['std_inner']:.4f} | "
            f"F1_train={fila['f1_train']:.4f} | "
            f"F1_outer={fila['f1_outer']:.4f} | gap={fila['gap']:.4f}"
        )
        print(msg, flush=True)
        monitor.registrar(msg)
        monitor.mostrar(msg, forzar=True)

    detalle = pd.DataFrame(resultados).sort_values("outer_fold").reset_index(drop=True)
    resumen = {
        "variante": variante,
        "n": len(y),
        "n_groups": int(df[GROUP_COL].nunique()),
        "f1_train_mean": float(detalle["f1_train"].mean()),
        "f1_nested_mean": float(detalle["f1_outer"].mean()),
        "f1_nested_std": float(detalle["f1_outer"].std(ddof=1)),
        "f1_nested_min": float(detalle["f1_outer"].min()),
        "f1_nested_max": float(detalle["f1_outer"].max()),
        "gap_mean": float(detalle["gap"].mean()),
    }
    return detalle, resumen

## 11. Ejecutar TPE20 con seguimiento y checkpoints

Tras `MODO="prueba"`, cambia a `"completo"`, reinicia el kernel y ejecuta desde el inicio. La corrida completa mantiene 5 Outer × 3 Inner × 20 trials. La salida visual sigue compacta; los CSV de pérdidas se guardan al completar cada Inner Fold. Los gráficos PNG del mejor trial se generan al finalizar cada Outer Fold.

**Para reanudar:** usa exactamente **este notebook**, mismo `MODO` y mismos datos. SQLite reencola los trials interrumpidos, los Inner Folds ya terminados no se repiten y `Trainer` reanuda desde el checkpoint más reciente de la época anterior. No borres `seguimiento/<modo>/` ni `checkpoints/<modo>/` mientras el experimento esté incompleto. Se eliminan checkpoints pesados solo después de confirmar que se guardaron resultados y curvas.


In [ ]:
if not EJECUTAR_NESTED:
    print("[DistilBETO] Nested CV desactivado. Ejecuta primero el pre-flight; luego cambia EJECUTAR_NESTED=True.")
    resultados_detalle = {}
    resultados_resumen = []
    resumen_nested = pd.DataFrame()
else:
    verificar_firma()
    total_fits = len(VARIANTES) * N_OUTER * (N_INNER * N_TRIALS + 1)
    monitor = Monitor(RUN_DIR, total=total_fits, ya_completados=contar_completados())
    resultados_detalle, resultados_resumen = {}, []
    try:
        for variante in VARIANTES:
            detalle, resumen = ejecutar_variante(variante, monitor)
            resultados_detalle[variante] = detalle
            resultados_resumen.append(resumen)
            csv_atomico(pd.DataFrame(resultados_resumen), RUN_DIR / "distilbeto_resumen_parcial.csv")
    except KeyboardInterrupt:
        monitor.registrar("INTERROMPIDO: los trials completos están en SQLite; revisar CSV de folds.")
        monitor.mostrar("Interrumpido; resultados parciales conservados", forzar=True)
        raise
    except Exception as exc:
        monitor.registrar(f"ERROR {type(exc).__name__}: {exc}")
        monitor.mostrar(f"Error: {type(exc).__name__}; consultar monitor.log", forzar=True)
        raise
    else:
        monitor.registrar("Nested CV completo para las variantes solicitadas")
        monitor.mostrar("Nested CV finalizado", forzar=True)

    resumen_nested = pd.DataFrame(resultados_resumen)
    print("\n[DistilBETO] RESUMEN NESTED CV")
    display(resumen_nested)
    for variante, detalle in resultados_detalle.items():
        print(f"\nDetalle {variante.upper()}")
        display(detalle.drop(columns=["best_params"]))

## 12. Guardar resultados de la nueva búsqueda

Durante la corrida ya se guardaron los CSV por Outer Fold y las bases SQLite por estudio. Esta celda consolida el resumen y detalle de las tres variantes al finalizar. **Conserva los CSV anteriores** para contrastar ambos procedimientos, sin elegir una variante del experimento mirando el test oficial.


In [ ]:
if not EJECUTAR_NESTED:
    print("No se guardan resultados Nested porque EJECUTAR_NESTED=False.")
elif MODO == "completo":
    csv_atomico(resumen_nested, RESULTS_DIR / "distilbeto_nestedcv_resumen.csv")
    for variante in VARIANTES:
        csv_atomico(
            resultados_detalle[variante], RESULTS_DIR / f"distilbeto_nestedcv_detalle_{variante}.csv"
        )
    print("Resultados completos disponibles en:", RESULTS_DIR)
else:
    print("Solo resultados de prueba, aislados en:", RUN_DIR)


## 13. Optimización final — **mantener desactivada mañana**

**No la ejecutes automáticamente al acabar Nested CV.** Primero estudia F1 externo, gaps y los historiales de los nuevos 20 trials. Si congelamos esta versión del procedimiento, esta sección lanzará 20 trials × 3 Inner Folds × 3 variantes = **180 entrenamientos adicionales**, seleccionará un preprocessing/configuración por variante usando **solo train** y guardará JSON. Eso se hace antes de abrir el test oficial y puede necesitar otra sesión.


In [ ]:
configs_finales = {}
if EJECUTAR_OPTIMIZACION_FINAL:
    if MODO != "completo":
        raise ValueError("La optimización final requiere MODO='completo'.")
    verificar_firma()
    for variante in VARIANTES:
        df = TRAIN_AGRUPADO[variante].copy()
        cache, y = cache_dataset(df)
        groups = df[GROUP_COL].astype(str).to_numpy()
        estudio = buscar_en_train(
            cache, y, groups, np.arange(len(y)), variante, "final",
            k_inner=3, n_trials=N_TRIALS, seed_cv=SEED, monitor=None,
        )
        cfg = estudio.best_params.copy()
        cfg["refit_epochs"] = int(estudio.best_trial.user_attrs["refit_epochs"])
        configs_finales[variante] = cfg
        print(f"{variante.upper()}: F1-CV final={estudio.best_value:.4f}; config={cfg}")
    (RESULTS_DIR / "distilbeto_config_final.json").write_text(
        json.dumps(configs_finales, indent=2, ensure_ascii=False), encoding="utf-8"
    )
else:
    print("Optimización final desactivada. Analiza primero los Outer Folds.")

## 14. Test oficial — **bloqueado**

No utilices los 600 tweets oficiales por variante hasta haber congelado el protocolo. Esta sección entrena 5 modelos nuevos por variante con semillas predeterminadas y reporta media ± desviación, sin seleccionar la mejor semilla según test. Se habilita únicamente con `EJECUTAR_TEST_OFICIAL=True` y la confirmación explícita definida en configuración. Si el nuevo experimento se descarta antes del test, conserva su resultado Nested CV como experimento separado; no mezcles trials de búsquedas distintas.


In [ ]:
def cargar_test(variante):
    ruta = DATA_DIR / f"test_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(ruta)
    df = pd.read_csv(ruta)
    req = [LABEL_COL, *TEXT_COLS.values()]
    if any(col not in df.columns for col in req):
        raise ValueError(f"Test {variante} no contiene todas las columnas esperadas.")
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    for col in TEXT_COLS.values():
        df[col] = df[col].fillna("").astype(str)
    if len(df) != 600:
        raise ValueError(f"Se esperan 600 ejemplos del test oficial {variante}, hay {len(df)}.")
    return df


def evaluar_semilla_final(train_df, test_df, cfg, seed, variante):
    prep = cfg["preprocessing"]
    # Protección adicional: textos diferentes pueden producir la misma secuencia
    # después de preprocessing + tokenizer uncased + truncamiento. No usar el
    # test para ELEGIR configuraciones: este control solo verifica integridad
    # cuando el protocolo ya está congelado y se habilitó la evaluación final.
    ids_train = tokenizer(train_df[TEXT_COLS[prep]].astype(str).tolist(),
                          truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]
    ids_test = tokenizer(test_df[TEXT_COLS[prep]].astype(str).tolist(),
                         truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]
    claves_train = {tuple(ids) for ids in ids_train}
    solapados = [i for i, ids in enumerate(ids_test) if tuple(ids) in claves_train]
    if solapados:
        raise ValueError(f"{variante}: {len(solapados)} entradas de test idénticas "
                         "a entradas de train después del tokenizer. "
                         "Audita y documenta antes de la evaluación final.")
    ds_train = DatasetTokenizado(train_df[TEXT_COLS[prep]], train_df[LABEL_COL])
    ds_test = DatasetTokenizado(test_df[TEXT_COLS[prep]], test_df[LABEL_COL])
    y_train = train_df[LABEL_COL].to_numpy(dtype=int)
    y_test = test_df[LABEL_COL].to_numpy(dtype=int)
    carpeta = FITS_DIR / variante / "test_final" / f"semilla{seed}"
    carpeta.mkdir(parents=True, exist_ok=True)
    resultado_file = carpeta / "resultado.json"
    cfg_sig = firma_config(cfg)
    if resultado_file.exists():
        previo = json.loads(resultado_file.read_text(encoding="utf-8"))
        if previo.get("config_firma") != cfg_sig:
            raise ValueError("Configuración final distinta a la guardada para la semilla")
        return {k: previo[k] for k in ("seed", "f1_macro", "accuracy",
                                     "precision_macro", "recall_macro")}
    cw = pesos_balanceados(y_train) if cfg["loss_mode"] == "balanced" else None
    model = crear_modelo(seed, cfg["seq_classif_dropout"])
    trainer = None
    try:
        args = argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=False,
                                       epochs=cfg["refit_epochs"])
        trainer = entrenador_silencioso(ClasificadorTrainer(
            model=model, args=args, train_dataset=ds_train,
            data_collator=data_collator, class_weights=cw,
        ))
        trainer.train(resume_from_checkpoint=ultimo_checkpoint_valido(carpeta))
        y_pred = predecir(trainer, ds_test)
        curva = extraer_curva(trainer.state.log_history)
        if not curva.empty:
            csv_atomico(curva, carpeta / "train_loss_final.csv")
        resultado = {
            "seed": int(seed),
            "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
            "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        }
        json_atomico({"config_firma": cfg_sig, **resultado}, resultado_file)
        limpiar_checkpoints_completados(carpeta)
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
    return resultado


In [ ]:
if EJECUTAR_TEST_OFICIAL:
    if MODO != "completo" or not EJECUTAR_OPTIMIZACION_FINAL:
        raise ValueError("Primero termina Nested CV y la optimización final en modo completo.")
    if CONFIRMACION_TEST != "TEST_OFICIAL_CONGELADO":
        raise ValueError("Bloqueado: confirma explícitamente la evaluación final.")
    datos_test = []
    for variante in VARIANTES:
        df_train = cargar_train(variante)
        df_test = cargar_test(variante)
        cfg = configs_finales[variante]
        for seed in SEMILLAS_FINALES:
            res = evaluar_semilla_final(df_train, df_test, cfg, seed, variante)
            res["variante"] = variante
            datos_test.append(res)
            print(f"{variante.upper()} seed {seed}: F1={res['f1_macro']:.4f}")
            csv_atomico(pd.DataFrame(datos_test), RESULTS_DIR / "distilbeto_test_5_semillas.csv")
    df_test_final = pd.DataFrame(datos_test)
    resumen = df_test_final.groupby("variante")["f1_macro"].agg(["mean", "std"]).reset_index()
    display(resumen)
    csv_atomico(resumen, RESULTS_DIR / "distilbeto_test_resumen.csv")
else:
    print("TEST OFICIAL BLOQUEADO; no se han cargado archivos de test.")

## 15. Lista de verificación para la sesión de seis horas

1. GPU disponible y entorno Jupyter correcto (`torch.cuda.is_available()`), sin otros entrenamientos simultáneos.
2. CSV `train_transformers_mx.csv`, `train_transformers_es.csv`, `train_transformers_cu.csv` presentes. Misma limpieza/etiquetas que la corrida anterior.
3. Ejecutar una prueba corta (`MODO="prueba"`) para comprobar el nuevo `seq_classif_dropout`; usa su carpeta separada.
4. Reiniciar kernel, poner `MODO="completo"`, verificar en pantalla **5 Outer, 3 Inner, 20 trials, 5 startup** y confirmar la carpeta nueva `resultados_distilbeto_tpe20`.
5. Dejar `EJECUTAR_OPTIMIZACION_FINAL=False`, `EJECUTAR_TEST_OFICIAL=False` y lanzar el Nested CV. Duración aproximada **4 h 40 min**, sujeta a variación.
6. Observar el panel y `monitor.log`. Cada trial queda registrado en SQLite, cada Outer Fold en CSV. Si se agota el tiempo, detener ordenadamente y reanudar otro día con los mismos datos/configuración.
7. Al finalizar, recopilar `distilbeto_nestedcv_resumen.csv`, `distilbeto_nestedcv_detalle_*.csv` y los historiales `trials_*.csv` de `seguimiento/completo/`.
8. Antes de cualquier optimización final o test oficial, analizar el experimento ampliado y decidir qué procedimiento conservar **sin utilizar test para comparar métodos**.

### Documentación

- [Optuna `n_startup_trials` en `TPESampler`](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html)
- [Optuna: guardar y recuperar estudios con SQLite](https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/001_rdb.html)
- [Hugging Face DistilBERT: `seq_classif_dropout`](https://huggingface.co/docs/transformers/en/model_doc/distilbert)
- [Trainer y callbacks de Hugging Face](https://huggingface.co/docs/transformers/main/trainer_callbacks)


## 16. Diagnóstico posterior de Optuna (sin entrenar)

Una vez terminada la corrida completa, esta celda recopila el historial de los 5 estudios Outer × 3 variantes = **15 estudios**, con hasta 20 trials completos cada uno (**300 trials** en total), e identifica en qué número de trial apareció la mejor F1 interna. También resume la frecuencia de los hiperparámetros de los Outer Folds. **Son diagnósticos descriptivos, no una segunda selección basada en Outer Test.**

In [ ]:
# Solo análisis de los resultados que YA están en CSV; no toca la GPU.
if MODO != "completo":
    print("Ejecuta este diagnóstico una vez acabada la corrida COMPLETA.")
else:
    historiales = []
    for variante in VARIANTES:
        for fold in range(1, N_OUTER + 1):
            ruta = RUN_DIR / f"trials_{variante}_outer{fold}.csv"
            if ruta.exists():
                temp = pd.read_csv(ruta)
                temp["variante"] = variante
                temp["outer_fold"] = fold
                historiales.append(temp)
    if historiales:
        df_historial = pd.concat(historiales, ignore_index=True)
        csv_atomico(df_historial, RESULTS_DIR / "distilbeto_optuna_historial_consolidado.csv")
        completados = df_historial[df_historial["state"] == "COMPLETE"].copy()
        print(f"Estudios detectados: {len(historiales)}/15; trials completos: {len(completados)}/300")
        # Best trial interno por estudio: no comparar esta puntuación entre datos diferentes.
        mejores = (
            completados.sort_values("value", ascending=False)
            .drop_duplicates(["variante", "outer_fold"])
            .sort_values(["variante", "outer_fold"])
        )
        columnas = ["variante", "outer_fold", "number", "value"] + [
            col for col in completados if col.startswith("params_")
        ]
        display(mejores[columnas].reset_index(drop=True))
        csv_atomico(mejores[columnas], RESULTS_DIR / "distilbeto_optuna_mejores_trials.csv")
    else:
        print("Todavía no hay historiales. Completa el Nested CV primero.")


## 17. Visualizar curvas de Loss y F1-Macro sin volver a entrenar


In [ ]:
# Modifica estos dos valores para revisar las curvas del mejor trial de un fold.
VARIANTE_GRAFICA = "mx"
OUTER_GRAFICA = 1

from IPython.display import Image

ruta_loss = CURVES_DIR / VARIANTE_GRAFICA / f"outer{OUTER_GRAFICA}_loss.png"
ruta_f1 = CURVES_DIR / VARIANTE_GRAFICA / f"outer{OUTER_GRAFICA}_f1.png"
ruta_csv = CURVES_DIR / VARIANTE_GRAFICA / f"outer{OUTER_GRAFICA}_media_por_epoca.csv"

if ruta_loss.exists() and ruta_f1.exists():
    print(f"[{VARIANTE_GRAFICA.upper()}][DistilBETO][Outer {OUTER_GRAFICA}] Curvas guardadas")
    display(Image(filename=str(ruta_loss)))
    display(Image(filename=str(ruta_f1)))
    display(pd.read_csv(ruta_csv))
else:
    print("Aún no están disponibles ambas curvas para ese Outer Fold.")
